In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# check for the GPU provided in the runtime
!nvidia-smi
# using quiet method for controlling the log
# for suppressing the colored errors and warning in the terminal
!pip install --quiet transformers==4.1.1
# pytorch lightning for smoother model training and data loading
!pip install --quiet https://github.com/PyTorchLightning/pytorch-lightning/releases/download/1.2.6/pytorch-lightning-1.2.6.tar.gz 
# using HuggingFace tokenizers
!pip install --quiet tokenizers==0.9.4
# Google's sentencepiece
!pip install --quiet sentencepiece==0.1.94
# mostly pl is used while doing complex model training
# import pytorch_lightning as pl
# print(pl.__version__)
# argparse makes it easier to write user friendly command line interfaces
import argparse
# package for faster file name matching
import glob
# makiing directories for data 
import os
# reading json files as the data is present in json files
import json
# time module for calculating the model runtime
import time
# Allows writing status messages to a file
import logging
# generate random float numbers uniformly
import random
# regex module for text 
import re
# module provides various functions which work on 
# iterators too produce complex iterators
from itertools import chain
from string import punctuation
# pandas for data manipulation
import pandas as pd
# numpy for array operations
import numpy as np
# PyTorch
import torch
# provides various classes representing file system paths
# with appropriate semantics
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
# import pytorch_lightning as pl
# splitting the data 
from sklearn.model_selection import train_test_split
# ANSII color formatting for ouput in terminal
from termcolor import colored
# wrapping paragraphs into string
import textwrap
# model checkpoints in pretrained model

'''
optimizer - AdamW
T5 Conditional Generator in which we'll give conditions
T5 tokenizer because it is fast
training the model without a learning rate
'''
from transformers import (
  AdamW,
  T5ForConditionalGeneration,
  T5Tokenizer,
  get_linear_schedule_with_warmup
)


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
#https://analyticsindiamag.com/guide-to-question-answering-system-with-t5-transformer/

In [ ]:
!pip install torchtext==0.10.0

In [ ]:
pip install pytorch_lightning

In [ ]:
# pip install -U torchtext -c pytorch

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
print(pl.__version__) 

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/FYP/MT5/Final Translated Dataset.csv")
# df.rename(columns = {'Questions':'question', 'Paragraphs':'context'}, inplace = True)

In [ ]:
df.rename(columns = {'Questions':'question', 'Paragraphs':'context', 'Answers':'answer_text'}, inplace = True)


In [ ]:
df.head()

In [ ]:
df.head()

In [ ]:
sample_question = df

In [ ]:
MODEL_NAME ='google/mt5-small' 
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)


In [ ]:
class BioQADataset(Dataset):
   def __init__(
       self,
       data:pd.DataFrame,
       tokenizer:T5Tokenizer,
       source_max_token_len: int = 396,
       target_max_token_len: int = 32,
       ):
     self.data =  data
     self.tokenizer =  tokenizer
     self.source_max_token_len =  source_max_token_len
     self.target_max_token_len =  target_max_token_len
   def __len__(self):
     return len(self.data)
   def __getitem__(self, index: int):
     data_row = self.data.iloc[index]
     source_encoding = tokenizer(
       data_row['question'],
       data_row['context'],
       max_length=self.source_max_token_len,
       padding='max_length',
       truncation="only_second",
       return_attention_mask=True,
       add_special_tokens=True,
       return_tensors="pt"
       )
     target_encoding = tokenizer(
       data_row['answer_text'],
       max_length=self.target_max_token_len,
       padding='max_length',
       truncation=True,
       return_attention_mask=True,
       add_special_tokens=True,
       return_tensors="pt"
       )
     labels = target_encoding['input_ids']
     labels[labels==0] = -100
     return dict(
         question=data_row['question'],
         context=data_row['context'],
         answer_text=data_row['answer_text'],
         input_ids=source_encoding["input_ids"].flatten(),
         attention_mask=source_encoding['attention_mask'].flatten(),
         labels=labels.flatten()
     )
sample_dataset = BioQADataset(df, tokenizer)
for data in sample_dataset:
  print("Question: ", data['question'])
  print("Answer text: ", data['answer_text'])
  print("Input_ids: ", data['input_ids'][:10])
  print("Labels: ", data['labels'][:10])
  break 

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.05)
train_df.shape,  val_df.shape

In [ ]:
class BioDataModule(pl.LightningDataModule):
   def __init__(
       self,
       train_df: pd.DataFrame,
       test_df: pd.DataFrame,
       tokenizer:T5Tokenizer,
       batch_size: int = 8,
       source_max_token_len: int = 396,
       target_max_token_len: int = 32,
       ):
     super().__init__()
     self.train_df = train_df
     self.test_df = test_df
     self.tokenizer = tokenizer
     self.batch_size = batch_size
     self.source_max_token_len = source_max_token_len
     self.target_max_token_len = target_max_token_len
   def setup(self):
     self.train_dataset = BioQADataset(
         self.train_df,
         self.tokenizer,
         self.source_max_token_len,
         self.target_max_token_len
         )
     self.test_dataset = BioQADataset(
     self.test_df,
     self.tokenizer,
     self.source_max_token_len,
     self.target_max_token_len
     )
   def train_dataloader(self):
     return DataLoader(
         self.train_dataset,
         batch_size=self.batch_size,
         shuffle=True,
         num_workers=4
         )
   def val_dataloader(self):
     return DataLoader(
         self.test_dataset,
         batch_size=self.batch_size,
         num_workers=4
         )
   def test_dataloader(self):
     return DataLoader(
         self.test_dataset,
         batch_size=1,
         num_workers=4
         )
BATCH_SIZE = 4
N_EPOCHS = 15
data_module = BioDataModule(train_df, val_df, tokenizer, batch_size=BATCH_SIZE)
data_module.setup() 

In [ ]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME, return_dict = True)
model.config

In [ ]:
class BioQAModel(pl.LightningModule):
   def __init__(self):
     super().__init__()
     self.model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME, return_dict=True)
   def forward(self, input_ids, attention_mask, labels=None):
     output = self.model(
         input_ids, 
         attention_mask=attention_mask,
         labels=labels)
     return output.loss, output.logits
   def training_step(self, batch, batch_idx):
     input_ids = batch['input_ids']
     attention_mask=batch['attention_mask']
     labels = batch['labels']
     loss, outputs = self(input_ids, attention_mask, labels)
     self.log("train_loss", loss, prog_bar=True, logger=True)
     return {"loss": loss, "predictions":outputs, "labels": labels}
   def validation_step(self, batch, batch_idx):
     input_ids = batch['input_ids']
     attention_mask=batch['attention_mask']
     labels = batch['labels']
     loss, outputs = self(input_ids, attention_mask, labels)
     self.log("val_loss", loss, prog_bar=True, logger=True)
     return loss
   def test_step(self, batch, batch_idx):
     input_ids = batch['input_ids']
     attention_mask=batch['attention_mask']
     labels = batch['labels']
     loss, outputs = self(input_ids, attention_mask, labels)
     self.log("test_loss", loss, prog_bar=True, logger=True)
     return loss
   def configure_optimizers(self):
     optimizer = AdamW(self.parameters(), lr=0.000005)
     return optimizer
model = BioQAModel()

In [ ]:
model = BioQAModel()
# To record the best performing model using checkpoint
checkpoint_callback = ModelCheckpoint(
    dirpath="/content/drive/MyDrive/Colab Notebooks/Datasets",
    filename="USquad-MRC-best-checkpoint",
    save_top_k=1,
    verbose=True,
    monitor="val_loss",
    mode="min"
)

trainer = pl.Trainer(
    #logger = logger,
    checkpoint_callback=checkpoint_callback,
    max_epochs=N_EPOCHS,
    gpus=1,
    progress_bar_refresh_rate = 30
)

In [ ]:

# trainer.fit(model, data_module)
# trainer.test()  # evaluate the model according to the last checkpoint 

In [ ]:

# trainer.fit(model, data_module)
# trainer.test()  # evaluate the model according to the last checkpoint 

In [ ]:

# trainer.fit(model, data_module)
# trainer.test()  # evaluate the model according to the last checkpoint 

In [ ]:

# trainer.fit(model, data_module)
# trainer.test()  # evaluate the model according to the last checkpoint 

In [ ]:
# model = BioQAModel.load_from_checkpoint("/content/drive/MyDrive/Colab Notebooks/Datasets/USquad-MRC-best-checkpoint.ckpt")
# trainer.fit(model, data_module)
# trainer.test()  # evaluate the model according to the last checkpoint 

In [ ]:
# model = BioQAModel.load_from_checkpoint("/content/drive/MyDrive/Colab Notebooks/Datasets/USquad-MRC-best-checkpoint.ckpt")
# trainer.fit(model, data_module)
# trainer.test()  # evaluate the model according to the last checkpoint 

In [ ]:
trained_model = BioQAModel.load_from_checkpoint("/content/drive/MyDrive/FYP/MT5/USquad-MRC-best-checkpoint.ckpt")
trained_model.freeze()

In [ ]:
# def generate_answer(question):
#    source_encoding=tokenizer(
#       #  "ک خود سے بیان کردہ &quot;جدید دور کی نسائی ماہر&quot;، بیونسے ایسے گانے تخلیق کرتی ہے جو اکثر محبت، رشتوں اور یک زوجگی کے ساتھ ساتھ خواتین کی جنسیت اور بااختیار بنانے کے موضوعات سے بھی متصف ہوتے ہیں۔ اسٹیج پر، اس کی متحرک، انتہائی کوریوگرافڈ پرفارمنس نے ناقدین کو عصری مقبول موسیقی کی بہترین تفریح کاروں میں سے ایک قرار دیا۔ 19 سال پر محیط اپنے کیریئر کے دوران، اس نے ایک سولو آرٹسٹ کے طور پر 118 ملین سے زیادہ ریکارڈز اور ڈیسٹینی چائلڈ کے ساتھ مزید 60 ملین ریکارڈز فروخت کیے ہیں، جس سے وہ اب تک کے سب سے زیادہ فروخت ہونے والے میوزک فنکاروں میں سے ایک ہیں۔ وہ 20 گریمی ایوارڈز جیت چکی ہیں اور ایوارڈ کی تاریخ میں سب سے زیادہ نامزد ہونے والی خاتون ہیں۔ امریکہ کی ریکارڈنگ انڈسٹری ایسوسی ایشن نے اسے 2000 کی دہائی کے دوران امریکہ میں سرٹیفائیڈ آرٹسٹ کے طور پر تسلیم کیا۔ 2009 میں، بل بورڈ نے انہیں دہائی کی ٹاپ ریڈیو گانے آرٹسٹ، 2000 کی ٹاپ فیمیل آرٹسٹ اور 2011 میں ان کی آرٹسٹ آف دی ملینیم کا نام دیا۔ ٹائم نے انہیں 2013 اور 2014 میں دنیا کے 100 بااثر لوگوں میں شامل کیا۔ فوربس میگزین انہیں 2015 کی سب سے طاقتور خاتون موسیقار کے طور پر بھی درج کیا گیا۔",
#       #   ,"بیونس نے کتنے گریمی ایوارڈز جیتے ہیں؟",
               
#        question["question"],
#        question['context'],
#        max_length = 396,
#        padding="max_length",
#        truncation="only_second",      
#        return_attention_mask=True,
#        add_special_tokens=True,
#        return_tensors="pt"
#    )
#    generated_ids = trained_model.model.generate(
#        input_ids=source_encoding["input_ids"],
#        attention_mask=source_encoding["attention_mask"],
#        num_beams=1,  # greedy search
#        max_length=80,
#        repetition_penalty=2.5,
#        early_stopping=True,
#        use_cache=True)
#    preds = [
#            tokenizer.decode(generated_id, skip_special_tokens=True, clean_up_tokenization_spaces=True)
#            for generated_id in generated_ids
#    ]
#    return "".join(preds)



In [ ]:
def generate_answer():
   source_encoding=tokenizer(
       "میرا نام سلمان ہے۔ میں لاہور میں رہتا ہوں۔"
        "میرا نام کیا ہے؟",
       
      #  "ک خود سے بیان کردہ &quot;جدید دور کی نسائی ماہر&quot;، بیونسے ایسے گانے تخلیق کرتی ہے جو اکثر محبت، رشتوں اور یک زوجگی کے ساتھ ساتھ خواتین کی جنسیت اور بااختیار بنانے کے موضوعات سے بھی متصف ہوتے ہیں۔ اسٹیج پر، اس کی متحرک، انتہائی کوریوگرافڈ پرفارمنس نے ناقدین کو عصری مقبول موسیقی کی بہترین تفریح کاروں میں سے ایک قرار دیا۔ 19 سال پر محیط اپنے کیریئر کے دوران، اس نے ایک سولو آرٹسٹ کے طور پر 118 ملین سے زیادہ ریکارڈز اور ڈیسٹینی چائلڈ کے ساتھ مزید 60 ملین ریکارڈز فروخت کیے ہیں، جس سے وہ اب تک کے سب سے زیادہ فروخت ہونے والے میوزک فنکاروں میں سے ایک ہیں۔ وہ 20 گریمی ایوارڈز جیت چکی ہیں اور ایوارڈ کی تاریخ میں سب سے زیادہ نامزد ہونے والی خاتون ہیں۔ امریکہ کی ریکارڈنگ انڈسٹری ایسوسی ایشن نے اسے 2000 کی دہائی کے دوران امریکہ میں سرٹیفائیڈ آرٹسٹ کے طور پر تسلیم کیا۔ 2009 میں، بل بورڈ نے انہیں دہائی کی ٹاپ ریڈیو گانے آرٹسٹ، 2000 کی ٹاپ فیمیل آرٹسٹ اور 2011 میں ان کی آرٹسٹ آف دی ملینیم کا نام دیا۔ ٹائم نے انہیں 2013 اور 2014 میں دنیا کے 100 بااثر لوگوں میں شامل کیا۔ فوربس میگزین انہیں 2015 کی سب سے طاقتور خاتون موسیقار کے طور پر بھی درج کیا گیا۔",
      #   "بیونس نے کتنے گریمی ایوارڈز جیتے ہیں؟",
               
      #  question["question"],
      #  question['context'],
       max_length = 396,
       padding="max_length",
       truncation="only_second",      
       return_attention_mask=True,
       add_special_tokens=True,
       return_tensors="pt"
   )
   generated_ids = trained_model.model.generate(
       input_ids=source_encoding["input_ids"],
       attention_mask=source_encoding["attention_mask"],
       num_beams=1,  # greedy search
       max_length=80,
       repetition_penalty=2.5,
       early_stopping=True,
       use_cache=True)
   preds = [
           tokenizer.decode(generated_id, skip_special_tokens=True, clean_up_tokenization_spaces=True)
           for generated_id in generated_ids
   ]
   return "".join(preds)

print(generate_answer())

In [ ]:

# # Import pandas package
# import pandas as pd
  
# # Define a dictionary containing employee data
# data = {'context':["ک خود سے بیان کردہ &quot;جدید دور کی نسائی ماہر&quot;، بیونسے ایسے گانے تخلیق کرتی ہے جو اکثر محبت، رشتوں اور یک زوجگی کے ساتھ ساتھ خواتین کی جنسیت اور بااختیار بنانے کے موضوعات سے بھی متصف ہوتے ہیں۔ اسٹیج پر، اس کی متحرک، انتہائی کوریوگرافڈ پرفارمنس نے ناقدین کو عصری مقبول موسیقی کی بہترین تفریح کاروں میں سے ایک قرار دیا۔ 19 سال پر محیط اپنے کیریئر کے دوران، اس نے ایک سولو آرٹسٹ کے طور پر 118 ملین سے زیادہ ریکارڈز اور ڈیسٹینی چائلڈ کے ساتھ مزید 60 ملین ریکارڈز فروخت کیے ہیں، جس سے وہ اب تک کے سب سے زیادہ فروخت ہونے والے میوزک فنکاروں میں سے ایک ہیں۔ وہ 20 گریمی ایوارڈز جیت چکی ہیں اور ایوارڈ کی تاریخ میں سب سے زیادہ نامزد ہونے والی خاتون ہیں۔ امریکہ کی ریکارڈنگ انڈسٹری ایسوسی ایشن نے اسے 2000 کی دہائی کے دوران امریکہ میں سرٹیفائیڈ آرٹسٹ کے طور پر تسلیم کیا۔ 2009 میں، بل بورڈ نے انہیں دہائی کی ٹاپ ریڈیو گانے آرٹسٹ، 2000 کی ٹاپ فیمیل آرٹسٹ اور 2011 میں ان کی آرٹسٹ آف دی ملینیم کا نام دیا۔ ٹائم نے انہیں 2013 اور 2014 میں دنیا کے 100 بااثر لوگوں میں شامل کیا۔ فوربس میگزین انہیں 2015 کی سب سے طاقتور خاتون موسیقار کے طور پر بھی درج کیا گیا۔"],
#         'question':["بیونس نے کتنے گریمی ایوارڈز جیتے ہیں؟"]}
#         # 'Address':['Delhi', 'Kanpur', 'Allahabad', 'Kannauj'],
#         # 'Qualification':['Msc', 'MA', 'MCA', 'Phd']}
  
# # Convert the dictionary into DataFrame 
# df = pd.DataFrame(data)
# newdata = df
  
# # select two columns
# sample_question = val_df.iloc[20]
# print(df['context', 'question'])
# print(newdata["context"])
# print(newdata["question"])
# print(generate_answer(newdata)) 

In [ ]:
sample_question = val_df.iloc[20]
print(sample_question["context"])
print(sample_question["question"])
print(sample_question["answer_text"])  # Label Answer
print(generate_answer(sample_question))  # Predicted answer

In [ ]:
sample_question = val_df.iloc[75]
print(sample_question["context"])
print(sample_question["question"])
print(sample_question["answer_text"])
print(generate_answer(sample_question)) 

In [ ]:
pip install rouge

In [ ]:
from rouge import Rouge
rouge = Rouge()


In [ ]:
rouge1 =[]
rouge2 =[]
rougel =[]
for x in range(len(val_df)):
  sample_question = val_df.iloc[x]
  h=generate_answer(sample_question)
  y=sample_question["answer_text"]
  scores = rouge.get_scores(h, y)

  rouge1.append(scores[0]["rouge-1"]['f'])
  rouge2.append(scores[0]["rouge-2"]['f'])
  rougel.append(scores[0]["rouge-l"]['f'])

print("F")
print("Rouge1", sum(rouge1)/len(rouge1))
print("Rouge2", sum(rouge2)/len(rouge2))
print("Rouge1", sum(rougel)/len(rougel))

In [ ]:

correct=0
exact=0
for x in range(len(val_df)):
  if(x%500==0):
    print (str(x)+'/'+ str(len(val_df)))
  sample_question = val_df.iloc[x]
  h=generate_answer(sample_question)
  y=sample_question["answer_text"]
  if h in y:
    correct = correct+1

  if h.strip() == y.strip():
    exact= exact + 1
print("Correct", correct, "Exact Match", exact , len(val_df))

In [ ]:
# 10 epochs
# F
# Rouge1 0.601074030167609
# Rouge2 0.2662376877937077
# Rouge1 0.6007236885407266

# 15 epoch
# F
# Rouge1 0.6117879486547141
# Rouge2 0.280017873845965
# Rouge1 0.6114023442829303

# 20 epochs
# F
# Rouge1 0.6147248799968577
# Rouge2 0.2817812569691026
# Rouge1 0.6143479683061136

#24
# Rouge1 0.7289561628669571
# Rouge2 0.3509014676796718
# Rouge1 0.7287494584288285

# +15
# changing error rate
Rouge1 0.7136821926525302
Rouge2 0.34666498146065633
Rouge1 0.7134818751492875